# Health Monitoring Data Analysis

This notebook demonstrates data analysis and visualization for the AI-Based Wearable Health Monitoring system.

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sensors.simulator import HealthSensorSimulator
from models.health_monitor import HealthMonitoringModel
from processing.data_processor import HealthDataProcessor

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Libraries imported successfully")

## 1. Generate Sample Data

In [ ]:
# Create simulator
simulator = HealthSensorSimulator(user_profile={
    'age': 35,
    'gender': 'M',
    'baseline_health': 'normal'
})

# Generate 24 hours of data (readings every 5 minutes)
print("Generating 24 hours of sensor data...")
data = simulator.generate_stream(duration_minutes=1440, interval_seconds=300)

print(f"Generated {len(data)} readings")
print("\nFirst few readings:")
data.head()

## 2. Data Statistics and Distribution

In [ ]:
# Basic statistics
print("Data Statistics:")
data[['heart_rate', 'spo2', 'temperature', 'systolic_bp', 'diastolic_bp']].describe()

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribution of Health Metrics', fontsize=16, fontweight='bold')

metrics = [
    ('heart_rate', 'Heart Rate (bpm)'),
    ('spo2', 'SpO2 (%)'),
    ('temperature', 'Temperature (°C)'),
    ('systolic_bp', 'Systolic BP (mmHg)'),
    ('diastolic_bp', 'Diastolic BP (mmHg)'),
    ('activity_level', 'Activity Level (%)')
]

for idx, (metric, label) in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    ax.hist(data[metric], bins=30, alpha=0.7, edgecolor='black')
    ax.set_xlabel(label)
    ax.set_ylabel('Frequency')
    ax.set_title(f'{label} Distribution')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Time Series Visualization

In [ ]:
# Plot vital signs over time
fig, axes = plt.subplots(4, 1, figsize=(15, 12))
fig.suptitle('24-Hour Health Monitoring Timeline', fontsize=16, fontweight='bold')

# Heart Rate
axes[0].plot(data['timestamp'], data['heart_rate'], color='red', linewidth=1)
axes[0].axhline(y=60, color='green', linestyle='--', alpha=0.5, label='Normal Range')
axes[0].axhline(y=100, color='green', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Heart Rate (bpm)')
axes[0].set_title('Heart Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# SpO2
axes[1].plot(data['timestamp'], data['spo2'], color='blue', linewidth=1)
axes[1].axhline(y=95, color='green', linestyle='--', alpha=0.5, label='Normal Threshold')
axes[1].set_ylabel('SpO2 (%)')
axes[1].set_title('Blood Oxygen Saturation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Temperature
axes[2].plot(data['timestamp'], data['temperature'], color='orange', linewidth=1)
axes[2].axhline(y=36.1, color='green', linestyle='--', alpha=0.5, label='Normal Range')
axes[2].axhline(y=37.2, color='green', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Temperature (°C)')
axes[2].set_title('Body Temperature')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Blood Pressure
axes[3].plot(data['timestamp'], data['systolic_bp'], color='purple', linewidth=1, label='Systolic')
axes[3].plot(data['timestamp'], data['diastolic_bp'], color='pink', linewidth=1, label='Diastolic')
axes[3].set_ylabel('BP (mmHg)')
axes[3].set_xlabel('Time')
axes[3].set_title('Blood Pressure')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Train ML Models

In [ ]:
# Initialize and train models
model = HealthMonitoringModel()

print("Training anomaly detection model...")
model.train_anomaly_detector(data, contamination=0.1)

print("\nTraining risk classification model...")
model.train_risk_classifier(data)

print("\n✓ Models trained successfully!")

## 5. Anomaly Detection Analysis

In [ ]:
# Detect anomalies in the dataset
anomalies = []
anomaly_scores = []

for idx, row in data.iterrows():
    is_anomaly, score = model.detect_anomaly(row.to_dict())
    anomalies.append(is_anomaly)
    anomaly_scores.append(score)

data['detected_anomaly'] = anomalies
data['anomaly_score'] = anomaly_scores

# Count anomalies
n_anomalies = sum(anomalies)
print(f"Detected {n_anomalies} anomalies out of {len(data)} readings")
print(f"Anomaly rate: {n_anomalies/len(data)*100:.2f}%")

In [ ]:
# Visualize anomalies
fig, ax = plt.subplots(figsize=(15, 6))

# Plot normal readings
normal = data[~data['detected_anomaly']]
ax.scatter(normal['timestamp'], normal['heart_rate'], 
           c='blue', alpha=0.5, s=20, label='Normal')

# Plot anomalies
anomaly_data = data[data['detected_anomaly']]
ax.scatter(anomaly_data['timestamp'], anomaly_data['heart_rate'], 
           c='red', alpha=0.8, s=100, marker='x', label='Anomaly', linewidths=2)

ax.set_xlabel('Time')
ax.set_ylabel('Heart Rate (bpm)')
ax.set_title('Anomaly Detection in Heart Rate')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Risk Assessment Analysis

In [ ]:
# Assess risk for all readings
risk_levels = []
risk_labels = []

for idx, row in data.iterrows():
    risk_level, _, risk_label = model.assess_risk(row.to_dict())
    risk_levels.append(risk_level)
    risk_labels.append(risk_label)

data['risk_level'] = risk_levels
data['risk_label'] = risk_labels

# Risk distribution
risk_counts = data['risk_label'].value_counts()
print("Risk Level Distribution:")
print(risk_counts)
print(f"\nPercentages:")
print(risk_counts / len(data) * 100)

In [ ]:
# Visualize risk distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart
colors = ['#10b981', '#3b82f6', '#f59e0b', '#ef4444']
ax1.pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax1.set_title('Risk Level Distribution')

# Time series of risk
ax2.plot(data['timestamp'], data['risk_level'], linewidth=1)
ax2.set_xlabel('Time')
ax2.set_ylabel('Risk Level')
ax2.set_title('Risk Level Over Time')
ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels(['Normal', 'Low', 'Moderate', 'High'])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Correlation matrix
metrics_for_corr = ['heart_rate', 'spo2', 'temperature', 'systolic_bp', 'diastolic_bp', 'activity_level']
correlation = data[metrics_for_corr].corr()

# Visualize correlation
plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Health Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
# Save analyzed data
output_file = '../data/analyzed_sensor_data.csv'
data.to_csv(output_file, index=False)
print(f"✓ Analyzed data saved to {output_file}")

# Save models
model.save_models('../models')
print("✓ Models saved to ../models/")